# Logistic Regression and LinearSVC with Regularization

**Artificially created data**

The following cell creates training data and three additional test samples. The data samples are defined in the same way as in https://github.com/amueller/introduction_to_ml_with_python/blob/main/mglearn/plot_knn_classification.py. Its not necessary to delve into the details of this code. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils import check_random_state, shuffle

def make_blobs(centers=2, random_state=4, n_samples=30):
    g = check_random_state(random_state)

    # fixed setup (2 centers, 2 features, std=1)
    centers = g.uniform(-10, 10, size=(centers, centers))
    neg = int(n_samples/2)
    pos = n_samples-neg
    n = [neg, pos]  # 30 samples split evenly

    X = np.vstack([centers[i] + g.normal(scale=1.0, size=(n[i], 2)) for i in range(2)])
    y = np.array([i for i in range(2) for _ in range(n[i])])

    X, y = shuffle(X, y, random_state=g)
    return X, y

# a carefully hand-designed dataset 
def make_forge():
    X, y = make_blobs(centers=2, random_state=4, n_samples=30)
    y[np.array([7, 27])] = 0 # color two points blue 
    mask = np.ones(len(X), dtype=bool) # make a boolean array with all trues
    mask[np.array([0, 1, 5, 26])] = 0 # remove this
    X, y = X[mask], y[mask]
    return X, y

def plotScatter(X, y, figsize):
    fig, axes = plt.subplots(1, 1, figsize=figsize) 
    discreteScatter(X,y,ax=axes)
    plt.legend(loc=4)
    plt.xlabel("First feature")
    plt.ylabel("Second feature")
    return plt

def discreteScatter(X,y,ax):
    markers = ['o', '^']
    for i, class_value in enumerate(np.unique(y)):
        ax.scatter(
            X[y == class_value, 0],
            X[y == class_value, 1],
            marker=markers[i % len(markers)],
            s=100,                    # 👈 größere Punkte
            edgecolor='black',        # 👈 schwarzer Rand
            label=f"Class {class_value}"
        )

X, y = make_forge()
plt = plotScatter(X, y, (10, 6))
plt.legend()
plt.show()
print("X.shape:", X.shape)



**Compare Logistic regression with Suport Vector Classifier**

In [ ]:
from sklearn.svm import LinearSVC
# if we get a warning that the iterations are not enough, we can increase them
# dual = auto selects the best algorithm for the ratio between #features / #samples
clf = LinearSVC()
clf.fit(X,y)  
print("Coefficients: ", clf.coef_)
print("Intercept:", clf.intercept_)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from mlxtend.plotting import plot_decision_regions

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for model, ax in zip([LinearSVC(C=1), LogisticRegression()], axes):
    clf = model.fit(X, y)
    plot_decision_regions(X, y,clf=clf,ax=ax, scatter_kwargs={'s': 100});
    ax.set_title(clf.__class__.__name__)
    ax.set_xlabel("Feature 0")
    ax.set_ylabel("Feature 1")
    
axes[0].legend()
plt.show()

<B>Regularization for SVC is done with C </B>

By default this does L2-Regularisation. In contrast to $\alpha$ heer we use the Rgularization Parameter C. <br>
Higher C leads to a more comples model (less regularisation) <br>
Lower C leads to a less complex model (more regularisation)<br>
The deault Value is 1 - LinearSVC(C=1)


In [ ]:
X, y = make_forge()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, C in zip(axes, [1e-2, 10, 1e3]):
        svm = LinearSVC(C=C, dual=False).fit(X, y) #Prefer dual=False when n_samples > n_features.
        plot_decision_regions(X, y,clf=svm,ax=ax, scatter_kwargs={'s': 100});
        ax.set_title("C = %2.2f" % C)
axes[0].legend(loc="best")
plt.show()